# 04 — Conditional q99 precursor discrimination

## Research question

Module 03 showed that q99 five-minute volatility onsets are often preceded by elevated volatility, activity, IV, wider spreads, and—where available—L2 instability.

Module 04 asks the harder question:

> **Among moments with a similar pre-existing native five-minute volatility environment, which Module 03 candidate variables still distinguish an imminent q99 onset from a non-q99 origin?**

## Explicit inheritance

This module inherits only the q99 event/onset definition and event-study utilities from Volatility Dynamics Module 03, plus a frozen candidate set selected from Module 03's own results. It reads no Module 03 output files.

It imports nothing from any other research branch.

## Native-volatility conditioning

Matching does **not** use rolling 30m/60m realized variance. For each origin, the volatility environment is the vector of exact single-bar five-minute squared-return observations at:

\[
-120,-60,-30,-15,-10,-5\text{ minutes}.
\]

Controls must be the same asset and exact five-minute UTC clock slot, then are chosen for closeness in this six-dimensional native-volatility history.


In [ ]:
from pathlib import Path
import json, os, sys, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 180)
pd.set_option('display.width', 280)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_q99_precursors as qp
from data_engineering.modeling import volatility_q99_conditional as qc

START = os.getenv('VD4_START', '').strip() or None
END = os.getenv('VD4_END', '').strip() or None
ASSET_FILTER = [x.strip().upper() for x in os.getenv('VD4_ASSETS', '').split(',') if x.strip()]
MAX_CONTROLS = int(os.getenv('VD4_MAX_CONTROLS', str(qc.DEFAULT_MAX_CONTROLS)))
MIN_CONTROLS = int(os.getenv('VD4_MIN_CONTROLS', str(qc.DEFAULT_MIN_CONTROLS)))
MIN_ENV_DIMS = int(os.getenv('VD4_MIN_ENV_DIMS', str(qc.DEFAULT_MIN_ENV_DIMS)))
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '04'
OUT.mkdir(parents=True, exist_ok=True)

print('Module 03 engine:', qp.VERSION)
print('Module 04 engine:', qc.VERSION)
print('Module 03 selftest:', qp.selftest())
print('Module 04 selftest:', qc.selftest())
print('window:', START or 'earliest available', '->', END or 'latest available')


## 1. Freeze the Module 03 candidate contract

The primary candidates are fixed before looking at Module 04 results. They represent the broad Module 03 findings: native range, trading activity, option IV levels, spread/book activity, and sparse L2-instability measures.

Rolling realized-volatility fields are included only as **environment diagnostics**. Weak Module 03 channels are retained as contrast checks.


In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
schema = con.execute('DESCRIBE time_series_feature_state').df()
available = set(schema['column_name'].astype(str))
contract = qc.candidate_contract(available)
display(contract)

primary = contract[(contract.role == 'primary_candidate') & contract.present]['feature'].tolist()
env_checks = contract[(contract.role == 'environment_check') & contract.present]['feature'].tolist()
contrast = contract[(contract.role == 'contrast_check') & contract.present]['feature'].tolist()
analysis_features = list(dict.fromkeys(primary + env_checks + contrast))
print('primary candidates:', primary)
print('environment checks:', env_checks)
print('contrast checks:', contrast)


## 2. Load only the target source and frozen Module 04 analysis fields


In [ ]:
def qident(name):
    return '"' + str(name).replace('"', '""') + '"'

select_cols = list(dict.fromkeys(['state_time','underlying','return_5m'] + analysis_features))
where = []
if START:
    where.append(f"state_time >= TIMESTAMPTZ '{pd.Timestamp(START, tz='UTC').isoformat()}'")
if END:
    end_ts = pd.Timestamp(END, tz='UTC') + pd.Timedelta(days=1)
    where.append(f"state_time < TIMESTAMPTZ '{end_ts.isoformat()}'")
if ASSET_FILTER:
    where.append('underlying IN (' + ','.join(repr(x) for x in ASSET_FILTER) + ')')
where_sql = '' if not where else ' WHERE ' + ' AND '.join(where)
sql = (
    'SELECT ' + ', '.join(qident(c) for c in select_cols)
    + ' FROM time_series_feature_state' + where_sql
    + ' ORDER BY underlying, state_time'
)
df = con.execute(sql).df()
con.close()
df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
df = df.reset_index(drop=True)
if df.empty:
    raise RuntimeError('No rows found.')
dupes = int(df.duplicated(['underlying','state_time']).sum())
if dupes:
    raise RuntimeError(f'Found {dupes} duplicate asset/timestamp rows.')
print(f'rows {len(df):,} | assets {sorted(df.underlying.unique())}')
print(df.state_time.min(), '->', df.state_time.max())


## 3. Recompute native q99 onsets

The q99 target is recomputed from `return_5m**2` inside Module 04. No Module 03 event file is read.


In [ ]:
events, event_meta = qp.identify_q99_onsets(df)
if events.empty:
    raise RuntimeError('No q99 event onsets found.')
origins = qp.build_origin_index(df, events)
display(event_meta)
print('asset-event onsets:', len(events), '| unique event timestamps:', events.event_time.nunique())


## 4. Build two control sets

**Clock-only controls** reproduce the Module 03 comparison: same asset, exact clock, nearest eligible dates.

**Conditional controls** keep same asset and exact clock but are selected by closeness of the native five-minute volatility history at -120/-60/-30/-15/-10/-5m.

Comparing the two tells us how much of each Module 03 candidate was simply a high-volatility-environment marker.


In [ ]:
clock_controls = qp.choose_controls(origins, events, max_controls=MAX_CONTROLS)
vol_controls, match_support = qc.choose_volatility_matched_controls(
    df, origins, events,
    max_controls=MAX_CONTROLS,
    min_env_dims=MIN_ENV_DIMS,
)
display(match_support[['n_eligible_same_clock','n_vol_comparable','n_controls_selected','median_selected_env_distance']].describe())
print('events with >= minimum selected controls:', int((match_support.n_controls_selected >= MIN_CONTROLS).sum()), '/', len(match_support))


## 5. Audit native-volatility match balance

A successful control set should substantially reduce the difference between event and control native log variance at the six matching lags.


In [ ]:
env = qc.build_native_vol_environment(df)
balance_clock = qc.environment_balance(env, events, clock_controls)
balance_clock['control_type'] = 'clock_only'
balance_conditional = qc.environment_balance(env, events, vol_controls)
balance_conditional['control_type'] = 'native_vol_matched'
balance = pd.concat([balance_clock, balance_conditional], ignore_index=True)
display(balance.pivot(index='lag_min', columns='control_type', values='median_event_minus_control_logvar').round(4))


## 6. Re-run candidate anatomy under both control definitions

Both studies use the extended ±120-minute window. Only negative lags can count as precursor evidence.


In [ ]:
relative_bars = range(-qc.ANATOMY_BARS, qc.ANATOMY_BARS + 1)
clock_scores = qp.percentile_scores(
    df, analysis_features, events, clock_controls, relative_bars,
    min_controls=MIN_CONTROLS,
)
conditional_scores = qp.percentile_scores(
    df, analysis_features, events, vol_controls, relative_bars,
    min_controls=MIN_CONTROLS,
)
if clock_scores.empty or conditional_scores.empty:
    raise RuntimeError('No usable candidate/control comparisons.')
clock_profiles = qp.summarize_profiles(clock_scores)
conditional_profiles = qp.summarize_profiles(conditional_scores)
conditional_by_asset = qp.summarize_by_asset(conditional_scores)
print('clock-only comparisons:', f'{len(clock_scores):,}')
print('conditional comparisons:', f'{len(conditional_scores):,}')


## 7. Conditional precursor rankings

The ranking horizon is now -120/-60/-30/-15/-10/-5 minutes. The same dependence corrections from Module 03 remain: simultaneous multi-asset events are collapsed by timestamp first, and candidates are penalized for inconsistent direction across assets or lags.


In [ ]:
clock_rankings = qp.precursor_rankings(clock_scores, rank_lags=qc.RANK_LAGS)
conditional_rankings = qp.precursor_rankings(conditional_scores, rank_lags=qc.RANK_LAGS)
comparison = qc.compare_rankings(clock_rankings, conditional_rankings, contract)
families = qc.family_summary(comparison)
display(comparison.round(4))
display(families.round(4))


## 8. Did the volatility diagnostics attenuate?

The rolling realized-volatility fields are not primary candidates here. They are a sanity check: if the native-volatility matching worked, their pre-event abnormality should be materially reduced relative to the clock-only comparison.


In [ ]:
diag = []
for f in env_checks:
    cr = clock_rankings[clock_rankings.feature == f]
    vr = conditional_rankings[conditional_rankings.feature == f]
    diag.append({
        'feature': f,
        'clock_abs_deviation': float(cr.abs_deviation_from_half.iloc[0]) if len(cr) else np.nan,
        'conditional_abs_deviation': float(vr.abs_deviation_from_half.iloc[0]) if len(vr) else np.nan,
        'clock_best_rel_min': int(cr.best_rel_min.iloc[0]) if len(cr) else np.nan,
        'conditional_best_rel_min': int(vr.best_rel_min.iloc[0]) if len(vr) else np.nan,
    })
diag = pd.DataFrame(diag)
diag['retention'] = diag.conditional_abs_deviation / diag.clock_abs_deviation
display(diag.round(4))


## 9. Plot the candidates that survive volatility conditioning best

The conditional percentile is what matters here. A trajectory near 0.5 after matching means the Module 03 anomaly was largely explained by the prior volatility environment.


In [ ]:
top_features = comparison.dropna(subset=['conditional_discovery_score']).head(10)['feature'].tolist()
for feature in top_features:
    cg = clock_profiles[clock_profiles.feature == feature].sort_values('rel_min')
    vg = conditional_profiles[conditional_profiles.feature == feature].sort_values('rel_min')
    if not len(vg):
        continue
    fig, ax = plt.subplots(figsize=(9,3))
    if len(cg):
        ax.plot(cg.rel_min, cg.mean_percentile, label='clock only', alpha=0.7)
    ax.plot(vg.rel_min, vg.mean_percentile, label='native-vol matched', linewidth=2)
    ax.axhline(0.5, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_ylim(0,1)
    ax.set_title(feature)
    ax.set_xlabel('minutes relative to q99 onset')
    ax.set_ylabel('mean percentile vs controls')
    ax.legend()
    plt.show()


## 10. Persist compact outputs


In [ ]:
contract.to_csv(OUT / '04_candidate_contract.csv', index=False)
event_meta.to_csv(OUT / '04_q99_thresholds_and_counts.csv', index=False)
events.to_csv(OUT / '04_q99_event_origins.csv', index=False)
match_support.to_csv(OUT / '04_vol_match_support.csv', index=False)
vol_controls.to_csv(OUT / '04_vol_matched_controls.csv', index=False)
balance.to_csv(OUT / '04_native_vol_environment_balance.csv', index=False)
clock_profiles.to_csv(OUT / '04_clock_only_profiles.csv', index=False)
conditional_profiles.to_csv(OUT / '04_vol_conditioned_profiles.csv', index=False)
conditional_by_asset.to_csv(OUT / '04_vol_conditioned_by_asset.csv', index=False)
clock_rankings.to_csv(OUT / '04_clock_only_rankings.csv', index=False)
conditional_rankings.to_csv(OUT / '04_vol_conditioned_rankings.csv', index=False)
comparison.to_csv(OUT / '04_candidate_comparison.csv', index=False)
families.to_csv(OUT / '04_family_summary.csv', index=False)
diag.to_csv(OUT / '04_environment_diagnostic_attenuation.csv', index=False)

top_set = set(comparison.dropna(subset=['conditional_discovery_score']).head(12)['feature'])
top_event_scores = conditional_scores[
    conditional_scores.feature.isin(top_set)
    & conditional_scores.rel_bar.isin(qc.RANK_LAGS)
].copy()
top_event_scores.to_csv(OUT / '04_top_conditional_event_scores.csv', index=False)

meta = {
    'version': qc.VERSION,
    'module': '04_conditional_q99_precursor_discrimination',
    'explicit_inheritance': 'q99 definition/utilities and frozen candidates from Volatility Dynamics Module 03 only; no Module 03 output files read',
    'database_relation': 'time_series_feature_state',
    'q99_recomputed_inside_module': True,
    'primary_candidates': primary,
    'matching_clock': 'same asset + exact 5m UTC slot',
    'volatility_environment': 'exact native 5m return^2 observations at -120,-60,-30,-15,-10,-5 minutes; log transformed for distance only',
    'rolling_volatility_used_for_matching': False,
    'anatomy_window_minutes': [-120,120],
    'ranking_lags_minutes': [-120,-60,-30,-15,-10,-5],
    'max_controls_per_event': MAX_CONTROLS,
    'min_controls_per_feature_event': MIN_CONTROLS,
    'min_native_vol_dimensions_for_match': MIN_ENV_DIMS,
    'no_predictive_model': True,
    'no_p_value_claim': True,
    'start': START or 'earliest_available',
    'end': END or 'latest_available',
    'n_rows': int(len(df)),
    'n_asset_event_onsets': int(len(events)),
    'n_unique_event_times': int(events.event_time.nunique()),
}
(OUT / '04_run_metadata.json').write_text(json.dumps(meta, indent=2))
print('wrote Module 04 outputs to', OUT)


## Interpretation boundary

A candidate that remains unusual after native-volatility matching has stronger evidence of **incremental precursor structure** beyond simply being in a high-volatility environment.

This still does not establish out-of-sample predictability. A later module must freeze surviving candidates and evaluate prospective q99 discrimination chronologically.
